# Course notebook — Reasoning and Tool Use

Understand when inference-time search, planning, verification, and external tools improve task outcomes.

Use Python 3.10 or later. All inputs are included and the workbook runs without API keys. Each lecture demonstration is a small model of the lecture mechanism. Run it, compare the result with your prediction, and use the section checkpoint to record what you learned.


## Search and planning


### Lecture 01 · Reasoning as Search

Separate recall from deliberate search and study decomposition, candidate generation, backtracking, and stopping.

**Predict before running.** Additional inference helps when the task benefits from exploring and rejecting alternatives. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Solve a constraint problem with explicit state search.


In [ ]:
"""Lab 7.1: Reasoning as Search"""

states = [("start", ["search US", "search CA"]), ("search US", ["merge"]), ("search CA", ["merge"])]
budget = 3
expanded = 0
agenda = ["start"]
while agenda and expanded < budget:
    node = agenda.pop(0)
    expanded += 1
    next_nodes = dict(states).get(node, [])
    agenda.extend(next_nodes)
print({"expanded": expanded, "remaining": agenda})


### Lecture 02 · Planning

Represent goals, prerequisites, steps, dependencies, state, uncertainty, and replanning without confusing a plausible plan with execution.

**Predict before running.** Plans are hypotheses about action sequences and must be updated by observations. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Build a planner that outputs a validated dependency graph.


In [ ]:
"""Lab 7.2: Planning"""

steps = {"fetch_US": [], "fetch_CA": [], "compare": ["fetch_US", "fetch_CA"], "draft": ["compare"]}
def valid_plan(completed):
    for step, prereqs in steps.items():
        if step in completed and not all(p in completed for p in prereqs):
            return False, step
    return True, "ok"
completed = {"fetch_US", "compare"}
print(valid_plan(completed))


**Section checkpoint.** Specify the goal, allowed actions, and stop condition for a small research workflow.


## Verification and tool contracts


### Lecture 03 · Verification and Critique

Use deterministic checks, tests, rubrics, critics, self-consistency, best-of-N, and external evidence.

**Predict before running.** Verification should exploit signals different from those used to generate the answer. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Generate several candidates and select with an independent verifier.


In [ ]:
"""Lab 7.3: Verification and Critique"""

candidates = [
    {"text": "Cap is 240", "cite_ok": True, "score": 0.7},
    {"text": "Cap is 300", "cite_ok": False, "score": 0.9},
]
def select(cands):
    passing = [c for c in cands if c["cite_ok"]]
    return max(passing, key=lambda c: c["score"]) if passing else None
print("selected:", select(candidates))


### Lecture 04 · Tools as Capability Boundaries

Design typed tools, schemas, descriptions, errors, timeouts, idempotency, permissions, and audit records.

**Predict before running.** Probabilistic intent must cross a deterministic, authorized boundary before effects occur. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Wrap a read-only API as a typed tool and fuzz its arguments.


In [ ]:
"""Lab 7.4: Tools as Capability Boundaries"""

def search_tool(query: str) -> dict:
    if not isinstance(query, str) or not query.strip():
        raise ValueError("query required")
    if len(query) > 200:
        raise ValueError("query too long")
    return {"results": [f"hit for {query!r}"]}
for q in ["budget policy", "", 123]:
    try:
        print(search_tool(q) if isinstance(q, str) else search_tool(str(q)))
    except ValueError as e:
        print({"error": str(e)})


**Section checkpoint.** Pass a complete plan and then swap two dependent steps. Compare the verifier's decisions.


## Integrations and reasoning cost


### Lecture 05 · MCP and Integration Protocols

Understand clients, servers, tools, resources, prompts, discovery, transport, authentication, and protocol security.

**Predict before running.** Protocols standardize capability exchange; they do not remove authorization or trust decisions. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Implement a small local MCP server and test a hostile client request.


In [ ]:
"""Lab 7.5: MCP and Integration Protocols"""

CHAPTER = "7.5"
print("chapter hook:", CHAPTER)
CLIENT_SCOPES = {"analyst": ["search_policy"]}
REQUEST = {"client": "analyst", "tool": "admin_delete"}
def authorize(client, tool):
    return tool in CLIENT_SCOPES.get(client, [])
print({"allowed": authorize(REQUEST["client"], REQUEST["tool"])})
print("---")
print("change one input above, predict output, re-run")


### Lecture 06 · Reasoning-System Economics

Balance accuracy, latency, token use, parallel candidates, tool calls, caches, failure rates, and task value.

**Predict before running.** Spend additional computation only where expected outcome improvement justifies it. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Plot quality and cost for single-pass, best-of-N, and verifier loops.


In [ ]:
"""Lab 7.6: Reasoning-System Economics"""

routes = [
    {"name": "single", "cost": 1, "quality": 0.78},
    {"name": "best3", "cost": 3, "quality": 0.86},
    {"name": "verify", "cost": 5, "quality": 0.91},
]
def pick(uncertainty, budget):
    opts = [r for r in routes if r["cost"] <= budget]
    if uncertainty < 0.3:
        return opts[0]
    return max(opts, key=lambda r: r["quality"])
print(pick(0.25, 4))
print(pick(0.8, 4))


**Section checkpoint.** List the minimum tool fields and estimate the budget for one verification retry.


## Course workshop

Construct a plan that contains every step but places the final review too early. Compare it with the valid plan.

Run the reference implementation, then run the checks below it.


In [ ]:
"""Generate candidate plans, verify constraints, and select the cheapest valid plan."""
PLANS = [
    ["read", "publish"],
    ["read", "draft", "review", "publish"],
    ["draft", "read", "publish"],
]


def verify(plan):
    required = {"read", "draft", "review", "publish"}
    if not required.issubset(plan): return False, "missing required step"
    positions = {step: plan.index(step) for step in required}
    if not positions["read"] < positions["draft"] < positions["review"] < positions["publish"]:
        return False, "invalid dependency order"
    return True, "valid"


valid = []
for plan in PLANS:
    ok, reason = verify(plan); print(plan, ok, reason)
    if ok: valid.append(plan)
print("selected=", min(valid, key=len))


### Verify and extend

Run the assertions. Then change one assumption and add a check for the predicted behavior.


In [ ]:
assert verify(PLANS[1])[0]
assert not verify(list(reversed(PLANS[1])))[0]
print('Workshop checks passed')


## Course project

Build a research workflow with a planner, typed tools, evidence store, verifier, and bounded recovery.

Keep your baseline, results, and one failure case so someone else can review the decision you made.
